# Part C: Regularized Linear Models
**Robust Regression Engine**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv(r"/mnt/user-data/uploads/Advanced_Regression_HousePrice_Dataset_3800_-_Advanced_Regression_HousePrice_Dataset_3800_csv.csv")

TARGET    = 'house_price_inr'
DROP_COLS = ['property_id', 'sale_date']
FEATURES  = [col for col in df.columns if col not in [TARGET] + DROP_COLS]

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("Data Ready")
print(X_train.shape)
print(X_test.shape)

## Task 9 — Ridge Regression (L2)

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

# Create Ridge Model
ridge = Ridge(alpha=1000)

# Train Model
ridge.fit(X_train_scaled, y_train)

# Prediction
y_pred_ridge = ridge.predict(X_test_scaled)

# Training Score
ridge_train = ridge.score(X_train_scaled, y_train)

# Testing Score
ridge_test = ridge.score(X_test_scaled, y_test)

# MSE
ridge_mse = mean_squared_error(y_test, y_pred_ridge)

# Print Results
print("===== Ridge Regression (L2) =====")
print("Training Score :", ridge_train)
print("Testing Score  :", ridge_test)
print("MSE            :", ridge_mse)
print("RMSE           :", np.sqrt(ridge_mse))
print("MAE            :", mean_absolute_error(y_test, y_pred_ridge))

# Coefficients Table
ridge_coef = pd.DataFrame({
    'Feature': FEATURES,
    'Coefficient': ridge.coef_
})
print("\nRidge Coefficients")
print(ridge_coef)

In [ ]:
# Coefficient Plot
ridge_coef_sorted = ridge_coef.sort_values('Coefficient')
plt.figure(figsize=(8, 5))
plt.barh(ridge_coef_sorted['Feature'], ridge_coef_sorted['Coefficient'], color='steelblue')
plt.title("Ridge Regression Coefficients")
plt.xlabel("Coefficient Value")
plt.tight_layout()
plt.show()

## Task 10 — Lasso Regression (L1)

In [ ]:
from sklearn.linear_model import Lasso

# Create Lasso Model
lasso = Lasso(alpha=1000, max_iter=50000)

# Train Model
lasso.fit(X_train_scaled, y_train)

# Prediction
y_pred_lasso = lasso.predict(X_test_scaled)

# Training Score
lasso_train = lasso.score(X_train_scaled, y_train)

# Testing Score
lasso_test = lasso.score(X_test_scaled, y_test)

# MSE
lasso_mse = mean_squared_error(y_test, y_pred_lasso)

# Print Results
print("===== Lasso Regression (L1) =====")
print("Training Score :", lasso_train)
print("Testing Score  :", lasso_test)
print("MSE            :", lasso_mse)
print("RMSE           :", np.sqrt(lasso_mse))
print("MAE            :", mean_absolute_error(y_test, y_pred_lasso))
print("Non-zero coefs :", np.sum(lasso.coef_ != 0), "/", len(lasso.coef_))

# Coefficients Table
lasso_coef = pd.DataFrame({
    'Feature': FEATURES,
    'Coefficient': lasso.coef_
})
print("\nLasso Coefficients")
print(lasso_coef)

## Coefficient Comparison: Ridge vs Lasso

In [ ]:
coef_df = pd.DataFrame({
    'Feature': FEATURES,
    'Ridge': ridge.coef_,
    'Lasso': lasso.coef_
})

coef_df.set_index('Feature', inplace=True)

coef_df.plot(kind='bar', figsize=(12, 6))
plt.title("Coefficient Comparison: Ridge vs Lasso")
plt.ylabel("Coefficient Value")
plt.xlabel("Features")
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()

## Task 11 — Tune Alpha using Cross-Validation

In [ ]:
alphas = np.logspace(0, 8, 100)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Best Ridge Alpha
ridge_cv = RidgeCV(alphas=alphas, cv=kf, scoring='r2')
ridge_cv.fit(X_train_scaled, y_train)
print("Best Ridge alpha :", ridge_cv.alpha_)

# Best Lasso Alpha
lasso_cv = LassoCV(alphas=alphas, cv=kf, max_iter=50000, random_state=42)
lasso_cv.fit(X_train_scaled, y_train)
print("Best Lasso alpha :", lasso_cv.alpha_)

In [ ]:
# Retrain with best alphas
best_ridge = Ridge(alpha=ridge_cv.alpha_)
best_ridge.fit(X_train_scaled, y_train)

best_lasso = Lasso(alpha=lasso_cv.alpha_, max_iter=50000)
best_lasso.fit(X_train_scaled, y_train)

print("===== Best Ridge =====")
print("Training Score :", best_ridge.score(X_train_scaled, y_train))
print("Testing Score  :", best_ridge.score(X_test_scaled, y_test))

print("\n===== Best Lasso =====")
print("Training Score :", best_lasso.score(X_train_scaled, y_train))
print("Testing Score  :", best_lasso.score(X_test_scaled, y_test))

## Task 12 — Compare Ridge vs Lasso: Error Curves & Coefficient Paths

In [ ]:
alphas_plot = np.logspace(0, 8, 60)
ridge_train_err, ridge_val_err = [], []
lasso_train_err, lasso_val_err = [], []

for a in alphas_plot:
    r = Ridge(alpha=a)
    cv_r = cross_val_score(r, X_train_scaled, y_train, cv=5, scoring='neg_mean_squared_error')
    r.fit(X_train_scaled, y_train)
    ridge_train_err.append(mean_squared_error(y_train, r.predict(X_train_scaled)))
    ridge_val_err.append(-cv_r.mean())

    l = Lasso(alpha=a, max_iter=50000)
    cv_l = cross_val_score(l, X_train_scaled, y_train, cv=5, scoring='neg_mean_squared_error')
    l.fit(X_train_scaled, y_train)
    lasso_train_err.append(mean_squared_error(y_train, l.predict(X_train_scaled)))
    lasso_val_err.append(-cv_l.mean())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].semilogx(alphas_plot, ridge_train_err, label='Train MSE', color='royalblue')
axes[0].semilogx(alphas_plot, ridge_val_err, label='CV Val MSE', color='tomato')
axes[0].set_xlabel('Alpha (log scale)')
axes[0].set_ylabel('MSE')
axes[0].set_title('Ridge (L2): Training vs Validation Error')
axes[0].legend()

axes[1].semilogx(alphas_plot, lasso_train_err, label='Train MSE', color='royalblue')
axes[1].semilogx(alphas_plot, lasso_val_err, label='CV Val MSE', color='tomato')
axes[1].set_xlabel('Alpha (log scale)')
axes[1].set_ylabel('MSE')
axes[1].set_title('Lasso (L1): Training vs Validation Error')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Coefficient Paths
ridge_coefs = [Ridge(alpha=a).fit(X_train_scaled, y_train).coef_ for a in alphas_plot]
lasso_coefs = [Lasso(alpha=a, max_iter=50000).fit(X_train_scaled, y_train).coef_ for a in alphas_plot]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, feat in enumerate(FEATURES):
    axes[0].semilogx(alphas_plot, [c[i] for c in ridge_coefs], label=feat)
    axes[1].semilogx(alphas_plot, [c[i] for c in lasso_coefs], label=feat)

axes[0].set_title('Ridge: Coefficient Paths (shrink but never zero)')
axes[1].set_title('Lasso: Coefficient Paths (features go to exactly 0!)')

for ax in axes:
    ax.set_xlabel('Alpha (log scale)')
    ax.set_ylabel('Coefficient Value')
    ax.legend(fontsize=7)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')

plt.tight_layout()
plt.show()